# SNOMED CT Concept Expansion: Hemochromatosis Case Study

This notebook demonstrates how to expand from a few hemochromatosis concepts to retrieve all related terms and their CUI codes using:
1. **MedCAT context similarity** - Finds conceptually similar terms
2. **SNOMED tree traversal** - Expands the concept hierarchy via parent-child relationships

## Use Case

This pattern is useful when you have a small set of target concepts and need to:
- Discover all related clinical terms
- Build comprehensive concept lists for projects
- Extract CUI codes for downstream analysis

## Setup

In [ ]:
import os
import sys
from typing import Dict, List, Tuple

import pandas as pd
from IPython.display import display as display_
from tqdm import tqdm

# Use relative path setup BEFORE importing snomed_methods_v1
CURRENT_DIR = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in locals()
    else os.getcwd()
)
project_root = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.snomed_methods.snomed_methods_v1 import SnomedRelations
from src.snomed_methods.snomed_term_lookup import create_term_lookup_from_directory

## Configuration

### Path Variables
- `SNOMED_DIR`: Base directory for SNOMED CT data (UK Clinical RF2)
- `SCT2_PATH`: Stated relationships file path for tree traversal
- `MEDCAT_MODEL_PACK_PATH`: MedCAT model pack for concept similarity (optional, warn repo not including normally)

In [ ]:
# Default paths using relative resolution from project root
CURRENT_DIR = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in locals()
    else os.getcwd()
)
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, ".."))

# Use environment variables or discovery functions
DEFAULT_SNOMED_DIR = os.environ.get(
    "UK_SNOMED_DIR", os.path.join(PROJECT_ROOT, "uk_sct2cl_42.2.0")
)

DEFAULT_SCT2_PATH = os.environ.get("SCT2_PATH")
if not DEFAULT_SCT2_PATH:
    import glob

    # Try to find the relationship file dynamically
    pattern = os.path.join(PROJECT_ROOT, "**/sct2_StatedRelationship_Full*.txt")
    matches = glob.glob(pattern, recursive=True)
    if matches:
        DEFAULT_SCT2_PATH = matches[0]
    else:
        # Last resort fallback
        DEFAULT_SCT2_PATH = os.path.join(
            PROJECT_ROOT,
            "uk_sct2cl_42.2.0",
            "SnomedCT_InternationalRF2_PRODUCTION_*",
            "Full",
            "Terminology",
            "sct2_StatedRelationship_Full_INT_*.txt",
        )

DEFAULT_UK_SNOMED_DIR = os.environ.get("UK_SNOMED_DIR")
if not DEFAULT_UK_SNOMED_DIR:
    import glob

    pattern = os.path.join(PROJECT_ROOT, "uk_*/*/SnomedCT_UKClinicalRF2*")
    matches = glob.glob(pattern)
    if matches:
        DEFAULT_UK_SNOMED_DIR = matches[0]
    else:
        DEFAULT_UK_SNOMED_DIR = os.path.join(
            PROJECT_ROOT, "uk_sct2cl_42.2.0", "SnomedCT_UKClinicalRF2_PRODUCTION_*"
        )

# MedCAT model pack path (optional - set to None if not available)
MEDCAT_MODEL_PACK_PATH = os.environ.get(
    "MEDCAT_MODEL_PACK_PATH"
)  # Set this to your model pack path or None

## Initialize Components

We need two main components:
- `SnomedTermLookup`: For term-to-CUI mapping and concept information
- `SnomedRelations`: For tree traversal via parent-child relationships

**Note on MedCAT:**
- To enable MedCAT similarity, set `MEDCAT_MODEL_PACK_PATH` above
- The model pack path must be configured via `MEDCAT_DHCAP02_PATH` environment variable before initializing `SnomedRelations`
- See Step 2 in the expansion function below for how this is handled

In [ ]:
# Initialize term lookup with UK Clinical RF2 data
lookup = create_term_lookup_from_directory(DEFAULT_UK_SNOMED_DIR)

# Initialize SnomedRelations for tree traversal (without MedCAT initially)
snomed_relations_obj = SnomedRelations(
    snomed_rf2_full_path=DEFAULT_SCT2_PATH, dhcap02=True  # Default configuration
)

## Define Starting Concepts

We start with a small set of hemochromatosis-related terms. The UK data uses British spelling.

In [ ]:
# Starting terms for hemochromatosis expansion
starting_terms = [
    "haemochromatosis",  # British spelling (correct)
    "iron overload",  # Related condition
    "hepatomegaly",  # Symptom
    "cirrhosis",  # Complication
]

# Find CUIs for starting terms using term lookup
starting_concepts = []
for term in starting_terms:
    results = lookup.find_concepts_by_term(term, top_n=1)
    if results:
        cui, matched_term = results[0]
        starting_concepts.append((cui, term))

## Function to Build Concept Expansion Results

This function combines:
1. **Tree traversal** - Expands via parent-child relationships from each starting CUI
2. **MedCAT similarity** (if model pack provided) - Finds conceptually similar terms

In [ ]:
def expand_concepts_from_terms(
    starting_concepts: List[Tuple[str, str]],
    n_tree_recursion: int = 5,
    medcat_topn: int = 20,
    model_pack_path: str = None,
) -> Dict:
    """
    Expand concepts from starting terms using tree traversal and MedCAT similarity.

    Args:
        starting_concepts: List of (cui, term) tuples to expand from
        n_tree_recursion: Recursion depth for SNOMED tree traversal
        medcat_topn: Number of similar concepts to retrieve via MedCAT
        model_pack_path: Path to MedCAT model pack (None = skip MedCAT)

    Returns:
        Dictionary with:
        - tree_expansion: Dict mapping starting CUI -> (codes, names) from
          tree traversal
        - medcat_similarities: Dict mapping starting CUI -> (codes, names)
          from similarity
        - all_concepts: Set of all unique Concept IDs found
        - concept_info: Dict mapping CUI -> concept information from term lookup
    """
    results = {
        "tree_expansion": {},
        "medcat_similarities": {},
        "all_concepts": set(),
        "concept_info": {},
    }

    # Step 1: Tree expansion for each starting concept
    for cui, _term in tqdm(starting_concepts, desc="Tree traversal"):
        try:
            codes, names = snomed_relations_obj.recursive_code_expansion(
                str(cui), n_recursion=n_tree_recursion, debug=False
            )
            results["tree_expansion"][cui] = (codes, names)
            if codes:
                results["all_concepts"].update(codes)
        except Exception:
            results["tree_expansion"][cui] = ([], [])

    # Step 2: MedCAT context similarity (if model pack provided)
    # Note: SnomedRelations requires path to be set via environment variable before init
    if model_pack_path and os.path.exists(model_pack_path):

        # Set environment variable for custom MedCAT model path
        os.environ["MEDCAT_DHCAP02_PATH"] = model_pack_path

        # Initialize SnomedRelations with MedCAT enabled using dhcap02 flag
        snomed_relations_with_medcat = SnomedRelations(
            snomed_rf2_full_path=DEFAULT_SCT2_PATH, medcat=True, dhcap02=True
        )

        for cui, _term in tqdm(starting_concepts, desc="MedCAT similarity"):
            try:
                codes, names = snomed_relations_with_medcat.get_medcat_cdb_most_similar(
                    cui, context_type="xxxlong", type_id_filter=[], topn=medcat_topn
                )
                results["medcat_similarities"][cui] = (codes, names)
                if codes:
                    results["all_concepts"].update(codes)
            except Exception:
                results["medcat_similarities"][cui] = ([], [])
    else:
        pass

    # Step 3: Build concept info dictionary
    for cui in tqdm(results["all_concepts"], desc="Fetching concept info"):
        try:
            info = lookup.getconcept_info(str(cui))
            results["concept_info"][cui] = {
                "preferred_name": info.get("preferred_name", "Unknown"),
                "all_names": info.get("all_names", []),
                "type_id": info.get("type_id", None),
            }
        except Exception:
            results["concept_info"][cui] = {
                "preferred_name": "Unknown",
                "all_names": [],
                "type_id": None,
            }

    return results

## Run Concept Expansion

In [ ]:
# Configure expansion parameters
TREE_RECURSION_DEPTH = 5
MEDCAT_TOP_N = 20

# Run expansion
expansion_results = expand_concepts_from_terms(
    starting_concepts=starting_concepts,
    n_tree_recursion=TREE_RECURSION_DEPTH,
    medcat_topn=MEDCAT_TOP_N,
    model_pack_path=MEDCAT_MODEL_PACK_PATH,  # Set this to your model pack path
)

## Summary Statistics

In [ ]:
# Calculate summary statistics
tree_total = sum(
    len(codes) if codes else 0
    for codes, _ in expansion_results["tree_expansion"].values()
)
medcat_total = sum(
    len(codes) if codes else 0
    for codes, _ in expansion_results["medcat_similarities"].values()
)

## Detailed Results by Starting Concept

In [ ]:
# Display detailed results for each starting concept
for cui, _term in starting_concepts:

    # Show tree expansion results
    tree_codes, tree_names = expansion_results["tree_expansion"].get(cui, ([], []))
    if tree_codes:
        for _i, (_code, _name) in enumerate(
            list(zip(tree_codes[:5], tree_names[:5])), 1
        ):
            pass
        if len(tree_codes) > 5:
            pass
    else:
        pass

    # Show MedCAT similarity results (if any)
    if expansion_results["medcat_similarities"]:
        medcat_codes, medcat_names = expansion_results["medcat_similarities"].get(
            cui, ([], [])
        )
        if medcat_codes:
            for _i, (_code, _name) in enumerate(
                list(zip(medcat_codes[:5], medcat_names[:5])), 1
            ):
                pass

## Export Concept List

Save all expanded concepts to CSV for use in downstream projects.

In [ ]:
# Build DataFrame from all expanded concepts
records = []

for cui in expansion_results["all_concepts"]:
    info = expansion_results["concept_info"].get(cui, {})
    records.append(
        {
            "cui": str(cui),
            "preferred_name": info.get("preferred_name", "Unknown"),
            "all_names": "; ".join(info.get("all_names", [])),
            "type_id": info.get("type_id"),
        }
    )

# Create and save DataFrame
CURRENT_DIR = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in locals()
    else os.getcwd()
)
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, ".."))
output_df = pd.DataFrame(records)
output_path = os.path.join(PROJECT_ROOT, "hemochromatosis_expansion_results.csv")
output_df.to_csv(output_path, index=False)

display_(output_df.head(10))

## Next Steps

The output CSV contains all expanded concepts with their CUIs and preferred names. This can be used in downstream projects for:
- Concept filtering in data processing pipelines
- Mapping clinical terms to standard codes
- Building classifier features based on SNOMED concept hierarchies